# CDSS-AB · Supplementary analysis example

Choose **Run → Run All Cells**. This runs two independent analyses, verifies their
outputs and displays the results below. No API key or GPU is needed.

**Expected:** 4 arithmetic questions · 3 fictional participants · 12 responses ·
8 correct answers · 3 answer revisions.

This is a software reproducibility example using synthetic inputs. It does not
reproduce clinical pilot results. Binder sessions are temporary: download your
results before leaving.

The interactive doctor interface is available from the JupyterLab launcher as
**CDSS-AB: doctor and administrator demo**, or from the main Binder link in the
README. This notebook analyses a separate fixed snapshot; it does not analyse
responses entered in the interactive exercise.


## 1. Run and verify

The verifier compares all 22 files between runs, checks 20 platform-independent
files against their fixed reference hashes, and checks the SQLite database's
complete schema and content plus its manifest against reviewed references.
It does not rewrite the references or claim macOS/Linux database binary equality.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import subprocess
import sys
import uuid
from IPython.display import FileLink, Markdown, display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'cdss-ab/scripts/run_binder_demo.py').is_file())
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
output = root / 'demo-results' / run_id
output.parent.mkdir(parents=True, exist_ok=True)
process = subprocess.run(
    [sys.executable, str(root / 'cdss-ab/scripts/run_binder_demo.py'),
     '--output-dir', str(output)],
    cwd=root, capture_output=True, text=True,
)
log = process.stdout + process.stderr
log_path = output.parent / (run_id + '.verification.txt')
log_path.write_text(log, encoding='utf-8')
if process.returncode:
    print(log)
    process.check_returncode()
print('\n'.join(log.splitlines()[-12:]))
print('Full verification log:', log_path.relative_to(root))


## 2. Inspect the result

The condition totals below come from the generated summary. The verifier also
checks database records and fixed expectations independently.


In [ ]:
summary = json.loads((output / 'run-1/summary.json').read_text(encoding='utf-8'))
overall = summary['overall']
display(Markdown(
    f"**{overall['n']} responses · {overall['correct']} correct · "
    f"{len(summary['answer_changes'])} answer revisions**"
))
rows = ['| Condition | Responses | Correct | Accuracy |', '| --- | ---: | ---: | ---: |']
for item in summary['by_condition']:
    rows.append(f"| {item['condition']} | {item['n']} | {item['correct']} | {item['accuracy']:.1%} |")
display(Markdown('\n'.join(rows)))


## 3. Download the results

The ZIP contains both runs and their verification log, including the JSON/CSV
summaries, SQLite database and original synthetic inputs. Download it now to keep
the results after the temporary Binder session ends.


In [ ]:
shutil.copyfile(log_path, output / 'verification.txt')
archive = Path(shutil.make_archive(str(output), 'zip', root_dir=output))
display(FileLink(str(archive.relative_to(Path.cwd())), result_html_prefix='Download results: '))
display(FileLink(str(log_path.relative_to(Path.cwd())), result_html_prefix='Verification log: '))
